# Queueing theory - E2/E2/1 example

An example of a queue in which both the interarrival times and the service times are Erlang 2 distributed. The simulated averages are compared with the M/M/1 queue with the same rates and with Kingman's approximation for G/G/1 queues.

The service station is composed of mix-ins, in the same way as objects in OpenTNSim. Because customers are generated by the arrival and service processes (rates, not a list of known customers), only the service station is created as an object.

This notebook provides an example how to set up a queueing simulation. Input that needs to be specified are:
* 'Lambda': the number of arrivals per hour
* 'Mu': the number of departures (services completed) per server per hour
* 'nr_resources': the number of servers of the service station (c in Kendall's notation)

To further characterise the queueing behaviour in the system the user has to specify the probability distributions that characterise the interarrival and service times. This notebook describes an E2/E2/1 queue (according to the Kendall notation for queueing systems) where both the interarrival times and the service times are assumed to be Erlang 2 distributed.

Currently implemented distributions are:
* 'M': Negative exponential distribution,
* 'Ek': Erlang k distributions (where k needs to be an integer number, e.g. E2, E3 etc).
* 'D': Deterministic lists with known input

NB: the calculation becomes more accurate when a larger number of arrivals (nr_arr) is simulated. Obviously there is a tradeoff between accuracy and calculation time.

In [1]:
# OpenQTSim mixins and utils
import openqtsim
from openqtsim.core import Identifiable, HasResource
from openqtsim.core import create_object, inspect_object
import pandas as pd
from openqtsim.visualizations import plot_gantt

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

This notebook is executed with OpenQTSim version v0.5.1


#### 0. Define input

In [2]:
example_title = "E2/E2/1"
Lambda = 8  # nr of arrivals per hour
Mu = 9  # nr of exits (services completed) per server per hour
nr_arr = 100_000
arrival_seed = 42  # random stream of the interarrival times
service_seed = 43  # independent random stream of the service times

#### 1. Create service station
Next we create a Server-object (our service station) out of the available mix-ins:

In [3]:
# make your preferred Server class out of available mix-ins.
Server = create_object(
    "Server",
    (
        Identifiable,  # allows to give the object a name and a random ID
        HasResource,   # number of servers (c in Kendall's notation)
    ),
)

We can inspect the object to determine which inputs are required and optional:

In [4]:
inspect_object(Server, show_parameter_table=True)

,parameter,class,default,status
0,name,Identifiable,,mandatory
1,nr_resources,HasResource,1,optional (unused)
2,id,Identifiable,None,optional (unused)


We define the required input and create the server object:

In [5]:
data_server = {
    "name": "Server",     # required by Identifiable
    "nr_resources": 1,    # optional for HasResource: one server
}

server = Server(**data_server)

#### 2. Define arrival and service processes
Customers are generated from Erlang 2 distributed interarrival and service times:

In [6]:
A = openqtsim.ArrivalProcess("E2", arr_rate=Lambda)
S = openqtsim.ServiceProcess("E2", srv_rate=Mu)

#### 3. Run simulation
The server object sets the number of servers of the queue.

In [7]:
%%time
q = openqtsim.Queue(A, S, station=server)
print('Start simulation of "{}" queue'.format(q.kendall_notation))

sim = openqtsim.Simulation(q, arrival_seed=arrival_seed, service_seed=service_seed)
sim.run(nr_arr)

Start simulation of "E2/E2/1/inf/inf/FIFO" queue
CPU times: user 3.69 s, sys: 581 ms, total: 4.27 s
Wall time: 3.87 s


In [8]:
summary = sim.get_stats()
print('Input IAT: {:.4f}'.format(1/Lambda))
print('Input ST: {:.4f}'.format(1/Mu))

Waiting time in units of service time: 3.7683
Observation horizon: 0 to 12538.8090 hours

Rho_system: system utilisation: 0.8864
Rho_server: server utilisation: 0.8864
P_0: probability nobody in the system: 0.1136

L_s: average nr of customers in the system: 4.2266
L_q: average nr of customers in the queue: 3.3402
W_s: average time spent in the system: 0.5300
W_q: average time spent in the queue: 0.4188

IAT: average inter arrival time: 0.1254
ST: average service time: 0.1111

Input IAT: 0.1250
Input ST: 0.1111


#### 4. Compare with M/M/1 and Kingman's approximation

There is no simple closed-form solution for the E2/E2/1 queue. A well-known approximation for a single-server G/G/1 queue is Kingman's formula:

$$W_q \approx \frac{c_a^2 + c_s^2}{2} \cdot \frac{\rho}{1-\rho} \cdot \frac{1}{\mu}$$

where $c_a$ and $c_s$ are the coefficients of variation of the interarrival and service times. For an Erlang k distribution $c^2 = 1/k$, so for E2 $c^2 = 0.5$ (for M $c^2 = 1$, which gives the exact M/M/1 result). $L_q = \lambda W_q$, $W_s = W_q + 1/\mu$ and $L_s = \lambda W_s$ follow from Little's law.

In [9]:
rho = Lambda / Mu
k_arrival, k_service = 2, 2  # Erlang shape parameters (E2)
ca2, cs2 = 1 / k_arrival, 1 / k_service

def gg1_measures(ca2, cs2):
    W_q = (ca2 + cs2) / 2 * rho / (1 - rho) / Mu
    W_s = W_q + 1 / Mu
    return {"IAT": 1 / Lambda, "ST": 1 / Mu, "Rho_server": rho,
            "L_s": Lambda * W_s, "L_q": Lambda * W_q, "W_s": W_s, "W_q": W_q}

kingman = gg1_measures(ca2, cs2)
mm1 = gg1_measures(1, 1)
simulated = {key: summary[key] for key in kingman}

comparison = pd.DataFrame({"M/M/1 (exact)": mm1, "E2/E2/1 Kingman (approx.)": kingman, "E2/E2/1 simulated": simulated})
comparison["simulated vs Kingman (%)"] = 100 * (comparison["E2/E2/1 simulated"] / comparison["E2/E2/1 Kingman (approx.)"] - 1)
comparison.round(4)

,M/M/1 (exact),E2/E2/1 Kingman (approx.),E2/E2/1 simulated,simulated vs Kingman (%)
IAT,0.1250,0.1250,0.1254,0.3065
ST,0.1111,0.1111,0.1111,0.0289
Rho_server,0.8889,0.8889,0.8864,-0.2807
L_s,8.0000,4.4444,4.2266,-4.9026
L_q,7.1111,3.5556,3.3402,-6.0580
W_s,1.0000,0.5556,0.5300,-4.6073
W_q,0.8889,0.4444,0.4188,-5.7664


Less variability in the interarrival and service times roughly halves the waiting time compared to the M/M/1 queue with the same utilisation. Kingman's formula is a heavy-traffic approximation; for this example it slightly overestimates the simulated waiting time. At a high utilisation ($\rho \approx 0.89$) successive waiting times are strongly correlated, so changing the seeds or the number of arrivals gives an impression of the remaining statistical uncertainty.

In [10]:
sim.plot_system_state()

In [11]:
sim.plot_IAT_ST()

#### 5. Inspect output

In [12]:
df, df_sys = sim.return_log()
df.head(10)

# Meaning of the columns:
# c_id: customer id
# IAT: interarrival time 
# ST: service time 
# AT: arrival time 
# TSB: time service begins
# TSE: time service ends 
# TCSS: time customer spends in the system
# TCWQ: time customer waits int he queue
# ITS: idle time server
# s_id: id of server assigned to customer

,c_id,name,length,maxwait,status,departure,IAT,ST,AT,TSB,TSE,TCSS,TCWQ,ITS,s_id
0,1,None,None,None,served,0.287157,0.141698,0.145459,0.141698,0.141698,0.287157,0.145459,0.000000,0.141698,1
1,2,None,None,None,served,0.366932,0.106507,0.079775,0.248204,0.287157,0.366932,0.118727,0.038952,0.000000,1
2,3,None,None,None,served,0.603168,0.272605,0.082359,0.520809,0.520809,0.603168,0.082359,0.000000,0.153877,1
3,4,None,None,None,served,0.718198,0.029460,0.115030,0.550269,0.603168,0.718198,0.167928,0.052899,0.000000,1
4,5,None,None,None,served,0.817941,0.128471,0.099743,0.678740,0.718198,0.817941,0.139201,0.039458,0.000000,1
5,6,None,None,None,served,0.846831,0.013340,0.028891,0.692080,0.817941,0.846831,0.154751,0.125861,0.000000,1
6,7,None,None,None,served,1.008974,0.208242,0.108651,0.900322,0.900322,1.008974,0.108651,0.000000,0.053491,1
7,8,None,None,None,served,1.048267,0.063246,0.039293,0.963568,1.008974,1.048267,0.084699,0.045405,0.000000,1
8,9,None,None,None,served,1.334214,0.336438,0.034207,1.300007,1.300007,1.334214,0.034207,0.000000,0.251740,1
9,10,None,None,None,served,1.501769,0.082961,0.118801,1.382968,1.382968,1.501769,0.118801,0.000000,0.048754,1


In [13]:
df_sys.head(10)

# Meaning of the columns:
# t: time of logging (if 'lambda' and 'mu' are specified as values per hour, the unit of 't' is hours)
# c_s: nr of customers in the system
# c_q nr of customers in the queue

,t,c_s,c_q
0,0.000000,0,0
1,0.141698,1,1
2,0.141698,1,0
3,0.248204,2,1
4,0.287157,1,1
5,0.287157,1,0
6,0.366932,0,0
7,0.520809,1,1
8,0.520809,1,0
9,0.550269,2,1


#### 6. Gantt chart: first 20 customers

Only the first 20 customers are shown. Orange spans arrival (`AT`) to service start (`TSB`); blue spans service start to departure (`TSE`). Hover to inspect the times in hours (and days) since simulation start.

In [14]:
first = df.sort_values(["AT", "c_id"]).head(20)
fig = plot_gantt(first, title=f"{example_title}: first {len(first)} of {len(df):,} customers "
                             f"(arrival seed {arrival_seed}, service seed {service_seed})")
fig.show()
first[["c_id", "AT", "TSB", "TSE", "TCWQ", "ST", "TCSS", "s_id"]]

,c_id,AT,TSB,TSE,TCWQ,ST,TCSS,s_id
0,1,0.141698,0.141698,0.287157,0.000000,0.145459,0.145459,1
1,2,0.248204,0.287157,0.366932,0.038952,0.079775,0.118727,1
2,3,0.520809,0.520809,0.603168,0.000000,0.082359,0.082359,1
3,4,0.550269,0.603168,0.718198,0.052899,0.115030,0.167928,1
4,5,0.678740,0.718198,0.817941,0.039458,0.099743,0.139201,1
5,6,0.692080,0.817941,0.846831,0.125861,0.028891,0.154751,1
6,7,0.900322,0.900322,1.008974,0.000000,0.108651,0.108651,1
7,8,0.963568,1.008974,1.048267,0.045405,0.039293,0.084699,1
8,9,1.300007,1.300007,1.334214,0.000000,0.034207,0.034207,1
9,10,1.382968,1.382968,1.501769,0.000000,0.118801,0.118801,1
